# Policy Gradients & On-Policy Dynamics

Now that we know how returns ($G_t$) and rewards work, we need the core mathematical mechanism: how do we actually update the neural network weights $\theta$ using these scalar rewards?

## 1. The Objective Function: What Are We Maximizing?

In supervised learning, we minimize cross-entropy loss against fixed targets.

In reinforcement learning, we maximize the expected return under the policy $\pi_\theta$:

$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]
$$

Where:

- $\tau = (s_0, a_0, s_1, a_1, \dots, s_T)$ is a full rollout trajectory, or the generated token sequence.
- $P(\tau \mid \theta) = P(s_0) \prod_{t=0}^T \pi_\theta(a_t \mid s_t)$ is the probability that policy $\pi_\theta$ generates this exact sequence.
- $R(\tau) = \sum_{t=0}^T r_t$ is the total return of the trajectory.

## 2. The Policy Gradient Theorem

To perform gradient ascent, we need the gradient $\nabla_\theta J(\theta)$:

$$
\nabla_\theta J(\theta)
= \nabla_\theta \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]
= \nabla_\theta \int P(\tau \mid \theta) R(\tau) \, d\tau
$$

### The problem

We cannot directly push the gradient $\nabla_\theta$ inside the integral because the probability distribution itself, $P(\tau \mid \theta)$, depends on $\theta$.

The environment transition and reward function are also effectively a black box and are not differentiable.

### The log-derivative trick

Recall that:

$$
\nabla f(x) = f(x) \nabla \log f(x)
$$

Applying this gives:

$$
\nabla_\theta P(\tau \mid \theta)
= P(\tau \mid \theta) \nabla_\theta \log P(\tau \mid \theta)
$$

Substituting back into the expectation yields:

$$
\nabla_\theta J(\theta)
= \mathbb{E}_{\tau \sim \pi_\theta}
\left[ \nabla_\theta \log P(\tau \mid \theta) R(\tau) \right]
$$

Expanding the trajectory into individual tokens $a_t$:

$$
\nabla_\theta J(\theta)
= \mathbb{E}_{\tau \sim \pi_\theta}
\left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot R(\tau) \right]
$$

## 3. The Intuition Behind REINFORCE

The update term is:

$$
\nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot R(\tau)
$$

### Case A: Successful generation

If the rollout succeeds and $R = +1.0$, the gradient pushes up the log-probability of the generated tokens.

That increases the probability of all tokens in that sequence.

### Case B: Failed generation

If the rollout fails and $R = 0.0$ or $-1.0$, the gradient pushes down the log-probability of the generated tokens.

That decreases the probability of all tokens in that sequence.

The log-probability gradient gives the direction in parameter space to make a token more likely, while the scalar reward controls the magnitude and sign of the update.

## 4. The Fatal Flaw of Pure REINFORCE: Variance and Baselines

Suppose every reward is positive, for example $R \in [10, 20]$.

- a mediocre response may get $R = 10$
- an exceptional response may get $R = 20$

Under pure REINFORCE, both responses get their probabilities increased because $R > 0$.

The model will eventually learn, but the gradient variance is enormous and training becomes unstable.

### The fix: advantage and baselines

Instead of multiplying by the raw return $R(\tau)$, we subtract a baseline $b(s_t)$:

$$
\nabla_\theta J(\theta)
= \mathbb{E} \left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot A(s_t, a_t) \right]
$$

Where the advantage is:

$$
A(s_t, a_t) = R(\tau) - b(s_t)
$$

- If an action produces an outcome better than average, $A > 0$, and its probability is increased.
- If an action produces an outcome worse than average, $A < 0$, and its probability is decreased.

Subtracting a baseline that does not depend on the action $a_t$ reduces variance without introducing bias into the gradient expectation.

## 5. On-Policy vs. Off-Policy Dynamics

### On-policy

The data used to compute the gradient $\nabla_\theta J(\theta)$ must be generated by the exact current parameters $\theta$.

Once you update $\theta \to \theta_{\text{new}}$, the old generated data is no longer valid and must be discarded.

### Why this matters for LLMs

Generating text requires running autoregressive forward passes across multiple GPUs.

In on-policy training, rollout generation is the main compute bottleneck:

1. generate a batch of samples,
2. compute the gradient,
3. update the weights,
4. discard the old samples,
5. generate new samples with the updated weights.

learning track 2:

Why Vanilla REINFORCE Sucks

Now let's make the environment actually require multiple decisions.

This is where the theory starts becoming interesting.

1. The problem with our toy environment

Previously:

S0
 ├── bad  → -1
 └── good → +1

The action immediately tells us whether we were correct.

Real RL is more like:

S0
 ↓
action
 ↓
S1
 ↓
action
 ↓
S2
 ↓
action
 ↓
S3
 ↓
reward

The agent might make 10 decisions before receiving one reward.

Now we have a serious question:

Which of those 10 actions actually deserves the credit?

That's the credit-assignment problem.